# Индивидуальный проект — начало...

## Данные + первый индекс

Цель пары:

1. выбрать датасет, который подойдёт для дальнейшего проекта
2. проверить, что в нём есть нужные типы полей
3. выбрать **одно уникальное поле**
4. реализовать свою `Hash Table`
5. добавить `resize` и `rehash`
6. использовать таблицу как индекс вида `уникальный ключ → row_id`
7. сравнить поиск полным проходом и поиск через индекс

На следующих занятиях проект будет постепенно расширяться новыми структурами и алгоритмами, поэтому к выбору датасета нужно подойти ответственно


---
# 1. Требования к датасету

Выберите датасет, с которым будете работать дальше в течение этого семестра

### Обязательные требования

Датасет должен содержать:

- желательно **не менее 100 000 строк**
- минимум **5 содержательных столбцов**
- минимум **одно уникальное поле** — идентификатор записи
- минимум **одно числовое поле**
- минимум **одно категориальное поле**
- **два поля, которые описывают связь между объектами**, и ещё одно числовое поле, которое характеризует эту связь. Например:
  - аэропорт отправления → аэропорт назначения, число — длительность полёта / цена билета / расстояние
  - отправитель → получатель, число — сумма перевода
  - пользователь → товар, число — цена / количество / оценка
  - пользователь → трек, число — количество прослушиваний / оценка / длительность прослушивания
  - команда 1 → команда 2, число — разница счёта / количество очков / длительность матча
  - пользователь 1 → пользователь 2, число — количество сообщений / взаимодействий / совместных действий
- минимум **одно поле, по которому значения можно упорядочить**:
  - число
  - дата/время
  - рейтинг
  - длительность и т. п.

### Желательно

Хорошо, если есть дата/время — это пригодится в следующих задачах

### Пример подходящего датасета

Датасет покупок в маркетплейсе:

```text
purchase_id   — уникальный идентификатор покупки
user_id       — идентификатор пользователя
product_id    — идентификатор товара
quantity      — количество купленного товара
price         — цена товара
category      — категория товара
created_at    — дата и время покупки
```

Почему подходит:

- `purchase_id` — уникальное поле
- `price`, `quantity` — числовые поля
- `category` — категориальное поле
- `created_at` — поле, значения которого можно упорядочить
- `user_id → product_id` — связь между двумя объектами: пользователь купил товар
- `quantity` или `price` — числовая характеристика этой связи

Одну строку можно прочитать так:

> Пользователь `user_id` купил товар `product_id` в количестве `quantity` по цене `price` в момент `created_at`.

**Сейчас использовать связь между объектами в коде не нужно. Она понадобится на следующих занятиях**


## Коротко опишите свой датасет

**Название / источник:**  
`MovieLens 10M от GroupLens`

**Что представляет одна строка:**  
`Информация о оценке фильма пользователем. С временем когда отзыв был оставлен`

**Размер датасета:**  
`1,000,000 строк и 8 столбцов.`

**Какие поля есть:**  
`rating_id, user_id, movie_id, rating, title, genres, created_at, main_genre`

**Какое поле является уникальным идентификатором:**  
`rating_id`

**Какое поле числовое:**  
`rating`

**Какое поле категориальное:**  
`main_genre`

**Какое поле можно использовать как упорядочиваемое:**  
`created_at`

**Какие два поля описывают связь между объектами:**  
`user_id → movie_id`

**Какое числовое поле характеризует эту связь:**  
`rating`


---
# 2. Загружаем данные

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from time import perf_counter

## Вариант A — загрузка файла с компа

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving movies.dat to movies.dat
Saving ratings.dat to ratings.dat


In [ ]:
ratings = pd.read_csv(
    "ratings.dat", sep=":", header=None,
    usecols=[0, 2, 4, 6],
    names=["user_id", "movie_id", "rating", "timestamp"],
)

movies = pd.read_csv(
    "movies.dat", sep="::", header=None, engine="python",
    names=["movie_id", "title", "genres"], encoding="latin-1",
)

df = ratings.merge(movies, on="movie_id", how="left")
df.insert(0, "rating_id", np.arange(len(df)))             # уникальный id записи
df["created_at"] = pd.to_datetime(df["timestamp"], unit="s")
df["main_genre"] = df["genres"].str.split("|").str[0]     # категориальное поле
df = df.drop(columns="timestamp")

df = df.sample(1_000_000, random_state=42).reset_index(drop=True)

# Пересчитываем уникальный id, чтобы он шел строго от 0 до 999 999 после перемешивания
df["rating_id"] = np.arange(len(df))

DATA_PATH = "data.csv"
df.to_csv(DATA_PATH, index=False)
print(df.shape)
df.head()

(1000000, 8)


,rating_id,user_id,movie_id,rating,title,genres,created_at,main_genre
0,0,958,1270,2.0,Back to the Future (1985),Adventure|Comedy|Sci-Fi,2003-04-13 03:23:55,Adventure
1,1,40310,364,3.5,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical,2006-05-28 01:33:24,Adventure
2,2,69359,2916,5.0,Total Recall (1990),Action|Adventure|Sci-Fi|Thriller,2001-02-02 04:46:56,Action
3,3,8467,296,3.5,Pulp Fiction (1994),Comedy|Crime|Drama,2006-12-05 20:16:29,Comedy
4,4,1883,1356,3.0,Star Trek: First Contact (1996),Action|Adventure|Sci-Fi|Thriller,1998-12-10 02:30:03,Action


## Вариант B — Google Drive

Используйте вместо предыдущей ячейки, если файл уже лежит на Google Drive

In [ ]:
print("Столбцы:")
for i, col in enumerate(df.columns):
    print(f"{i:>2}. {col:<30} dtype={df[col].dtype}")

print()

if len(df) >= 100_000:
    print("Размер датасета подходит")
else:
    print(f"В датасете {len(df):,} строк. Подумай над своими действиями...")

Столбцы:
 0. rating_id                      dtype=int64
 1. user_id                        dtype=int64
 2. movie_id                       dtype=int64
 3. rating                         dtype=float64
 4. title                          dtype=object
 5. genres                         dtype=object
 6. created_at                     dtype=datetime64[ns]
 7. main_genre                     dtype=object

Размер датасета подходит


---
# 3. Выбираем уникальное поле

Сегодня строим индекс только для **уникального ключа**

Примеры:

```text
идентификатор покупки
идентификатор транзакции
идентификатор поездки
идентификатор рейса
```

Для такого поля будем хранить:

```text
уникальный ключ → row_id
```

Например:

```text
purchase_id 837291 → строка 41827
```


In [ ]:
UNIQUE_FIELD = "rating_id"

assert UNIQUE_FIELD in df.columns, (
    f"Столбца {UNIQUE_FIELD!r} нет. "
    f"Доступные столбцы: {list(df.columns)}"
)

series = df[UNIQUE_FIELD]

print("Поле:", UNIQUE_FIELD)
print("Количество строк:", len(series))
print("Уникальных значений:", series.nunique(dropna=False))
print("Пропущенных значений:", series.isna().sum())
print("Поле уникально:", series.is_unique)

Поле: rating_id
Количество строк: 1000000
Уникальных значений: 1000000
Пропущенных значений: 0
Поле уникально: True


### Требование

Для основной части работы поле должно быть:

- **уникальным**
- **без пропущенных значений**

Если поле не проходит проверку, выберите другое

Если в датасете нет естественного уникального идентификатора — обсудите это с преподавателем


In [ ]:
assert series.is_unique, (
    "Выбранное поле не уникально"
    "Для сегодняшнего задания выберите уникальный идентификатор"
)

assert not series.isna().any(), (
    "В уникальном поле есть пропущенные значения"
    "Выберите поле без пропусков"
)

keys = series.tolist()
print("Уникальный ключ выбран")


Уникальный ключ выбран


---
# 4. Поиск без индекса

Сначала реализуем обычный поиск полным проходом

Функция должна вернуть `row_id` найденной записи или `None`

In [ ]:
def linear_search(values, target):
    for i in range(len(values)):
      if values[i] == target: return i
    return None

In [ ]:
TARGET = keys[len(keys) // 2]

row_id = linear_search(keys, TARGET)

print("TARGET:", TARGET)
print("row_id:", row_id)

if row_id is not None:
    display(df.iloc[[row_id]])

TARGET: 500000
row_id: 500000


,rating_id,user_id,movie_id,rating,title,genres,created_at,main_genre
500000,500000,15003,2147,2.0,"Clan of the Cave Bear, The (1986)",Drama,2000-11-20 05:41:30,Drama


### Ответьте

**Временная сложность в худшем случае:**  
`O(n)`

**Дополнительная память:**  
`O(1)`

**Почему при миллионах записей такой поиск может быть проблемой:**  
`В связи с линейной сложностью.`

---
# 5. Реализуем Hash Table

Идея:

```text
key
 ↓
hash(key)
 ↓
hash(key) % capacity
 ↓
bucket
```

Разные ключи могут попасть в один `bucket` — это **коллизия**

Коллизии в этой работе решаем методом **separate chaining**:

```text
bucket → [(key1, value1), (key2, value2), ...]
```

## Обязательное расширение таблицы

Таблица начинается с небольшой вместимости:

```text
capacity = 8
```

Коэффициент заполнения:

```text
load_factor = size / capacity
```

В нашей реализации используем порог:

```text
MAX_LOAD_FACTOR = 0.75
```

Если добавление **нового** ключа приведёт к:

```text
(size + 1) / capacity > 0.75
```

таблица должна:

1. увеличить `capacity` в 2 раза
2. создать новые пустые `buckets`
3. заново распределить по ним **все старые пары**
4. после этого добавить новый элемент

Это называется **rehash**

Почему нельзя просто увеличить список `buckets`?

Потому что номер bucket вычисляется так:

```text
hash(key) % capacity
```

После изменения `capacity` тот же ключ может получить уже другой номер bucket

> `0.75` — порог, который мы используем в нашей учебной реализации. Это не универсальное правило для всех Hash Table

### Ограничение

Внутри `MyHashMap` нельзя использовать:

- `dict`
- `set`
- `defaultdict`


In [ ]:
class MyHashMap:
    MAX_LOAD_FACTOR = 0.75

    def __init__(self, capacity=8):
        self.capacity = capacity
        self.size = 0
        self.buckets = [[] for _ in range(capacity)]

    def _index(self, key):
        return hash(key) % self.capacity

    def load_factor(self):
        return self.__len__() / self.capacity

    def _resize(self):
        old_buckets = self.buckets

        self.capacity *= 2
        self.buckets = [[] for _ in range(self.capacity)]

        for bucket in old_buckets:
            for key, value in bucket:
                idx = self._index(key)
                self.buckets[idx].append([key, value])

    def put(self, key, value):
        idx = self._index(key)

        for pair in self.buckets[idx]:
            if pair[0] == key:
                pair[1] = value
                return

        if (self.size + 1) / self.capacity > self.MAX_LOAD_FACTOR:
            self._resize()
            idx = self._index(key)

        self.buckets[idx].append([key, value])
        self.size += 1

    def get(self, key):
        idx = self._index(key)

        for k, v in self.buckets[idx]:
            if k == key:
                return v

        return None

    def contains(self, key):
        idx = self._index(key)

        for k, _ in self.buckets[idx]:
            if k == key:
                return True

        return False

    def remove(self, key):
        idx = self._index(key)
        bucket = self.buckets[idx]

        for i in range(len(bucket)):
            if bucket[i][0] == key:
                bucket.pop(i)
                self.size -= 1

                return True

        return False

    def __len__(self):
        return self.size


---
# 6. Проверяем реализацию

In [ ]:
def check(name, condition):
    print(("СУПЕР!!!!" if condition else "МДАААА"), name)

m = MyHashMap(4)

m.put("a", 10)
check("put/get", m.get("a") == 10)

m.put("a", 20)
check("обновление существующего ключа", m.get("a") == 20)
check("size после update не вырос", len(m) == 1)

check("contains", m.contains("a"))
check("contains missing", not m.contains("missing"))

check("remove", m.remove("a") is True)
check("ключ удалён", not m.contains("a"))
check("повторный remove", m.remove("a") is False)

# Проверяем, что обновление существующего ключа
# не должно само по себе вызывать resize
m = MyHashMap(4)
m.put("a", 1)
m.put("b", 2)
m.put("c", 3)          # load factor = 0.75

capacity_before = m.capacity
m.put("a", 100)        # update, а не новый ключ

check("update не вызывает resize", m.capacity == capacity_before)
check("значение после update изменилось", m.get("a") == 100)


СУПЕР!!!! put/get
СУПЕР!!!! обновление существующего ключа
СУПЕР!!!! size после update не вырос
СУПЕР!!!! contains
СУПЕР!!!! contains missing
СУПЕР!!!! remove
СУПЕР!!!! ключ удалён
СУПЕР!!!! повторный remove
СУПЕР!!!! update не вызывает resize
СУПЕР!!!! значение после update изменилось


## Проверка коллизий

Чтобы `resize` не мешал эксперименту, возьмём `capacity = 8`

Ключи:

```text
1, 9, 17, 25
```

при `capacity = 8` имеют одинаковый остаток при делении на 8 и попадают в один bucket

Все значения всё равно должны корректно находиться


In [ ]:
m = MyHashMap(8)

for key in [1, 9, 17, 25]:
    m.put(key, key * 10)

check("collision 1", m.get(1) == 10)
check("collision 9", m.get(9) == 90)
check("collision 17", m.get(17) == 170)
check("collision 25", m.get(25) == 250)
check("resize ещё не нужен", m.capacity == 8)

m.remove(9)

check("9 удалён", m.get(9) is None)
check("1 остался", m.get(1) == 10)
check("17 остался", m.get(17) == 170)
check("25 остался", m.get(25) == 250)


СУПЕР!!!! collision 1
СУПЕР!!!! collision 9
СУПЕР!!!! collision 17
СУПЕР!!!! collision 25
СУПЕР!!!! resize ещё не нужен
СУПЕР!!!! 9 удалён
СУПЕР!!!! 1 остался
СУПЕР!!!! 17 остался
СУПЕР!!!! 25 остался


## Проверка `resize` и `rehash`

При `capacity = 4`:

```text
3 / 4 = 0.75
```

Три элемента еще помещаются без расширения

Но добавление четвертого дало бы:

```text
4 / 4 = 1.0
```

Поэтому **перед добавлением четвёртого нового ключа** таблица должна расшириться:

```text
4 → 8
```

После `resize` все старые элементы должны по-прежнему находиться


In [ ]:
m = MyHashMap(4)

m.put(1, "A")
m.put(2, "B")
m.put(3, "C")

check("после 3 элементов capacity = 4", m.capacity == 4)
check("load factor = 0.75", abs(m.load_factor() - 0.75) < 1e-9)

m.put(4, "D")

check("при добавлении 4-го элемента capacity стала 8", m.capacity == 8)
check("1 не потерялся после rehash", m.get(1) == "A")
check("2 не потерялся после rehash", m.get(2) == "B")
check("3 не потерялся после rehash", m.get(3) == "C")
check("4 добавился", m.get(4) == "D")
check("load factor снова <= 0.75", m.load_factor() <= MyHashMap.MAX_LOAD_FACTOR)


СУПЕР!!!! после 3 элементов capacity = 4
СУПЕР!!!! load factor = 0.75
СУПЕР!!!! при добавлении 4-го элемента capacity стала 8
СУПЕР!!!! 1 не потерялся после rehash
СУПЕР!!!! 2 не потерялся после rehash
СУПЕР!!!! 3 не потерялся после rehash
СУПЕР!!!! 4 добавился
СУПЕР!!!! load factor снова <= 0.75


---
# 7. Строим индекс на своем датасете

Теперь используем собственную Hash Table как реальный индекс:

```text
уникальный ключ → row_id
```

In [ ]:
def build_hash_index(values):
    index = MyHashMap()

    for row_id, key in enumerate(values):
        index.put(key, row_id)

    return index


In [ ]:
hash_index = build_hash_index(keys)

print("Индекс построен")
print("Количество ключей:", len(hash_index))
print("Итоговая capacity:", hash_index.capacity)
print(f"Load factor: {hash_index.load_factor():.4f}")


Индекс построен
Количество ключей: 1000000
Итоговая capacity: 2097152
Load factor: 0.4768


In [ ]:
TARGET = keys[len(keys) // 2]

row_id = hash_index.get(TARGET)

print("TARGET:", TARGET)
print("row_id:", row_id)

if row_id is not None:
    display(df.iloc[[row_id]])

TARGET: 500000
row_id: 500000


,rating_id,user_id,movie_id,rating,title,genres,created_at,main_genre
500000,500000,15003,2147,2.0,"Clan of the Cave Bear, The (1986)",Drama,2000-11-20 05:41:30,Drama


---
# 8. Сравниваем два способа поиска

In [ ]:
test_positions = np.linspace(
    0,
    len(keys) - 1,
    num=min(30, len(keys)),
    dtype=int
)

test_keys = [keys[i] for i in test_positions]

start = perf_counter()
for key in test_keys:
    linear_search(keys, key)
linear_ms = (perf_counter() - start) * 1000

start = perf_counter()
for key in test_keys:
    hash_index.get(key)
hash_ms = (perf_counter() - start) * 1000

print(f"Запросов: {len(test_keys)}")
print(f"Linear search: {linear_ms:.3f} ms")
print(f"Hash lookup:   {hash_ms:.3f} ms")

if hash_ms > 0:
    print(f"Отношение времени: {linear_ms / hash_ms:.2f}x")

Запросов: 30
Linear search: 1392.055 ms
Hash lookup:   0.308 ms
Отношение времени: 4512.55x


## Стоимость построения индекса

In [ ]:
start = perf_counter()
tmp_index = build_hash_index(keys)
build_ms = (perf_counter() - start) * 1000

print(f"Build hash index: {build_ms:.3f} ms")

Build hash index: 6088.511 ms


---
# 9. Итог сегодняшней версии проекта

Заполните:

**Датасет:**  
`MovieLens 10M (GroupLens), случайная выборка`

**Размер:**  
`1 000 000 строк, 8 столбцов`

**Уникальное поле:**  
`rating_id`

**Что хранит индекс:**  

`rating_id → row_id
`

**Поиск без индекса:**  
`linear_search: проход по списку ключей и сравнение каждого элемента с искомым, пока не найдём.`

**Сложность поиска без индекса:**  
`O(n) по времени, O(1) по дополнительной памяти. В худшем случае ключ в конце или его нет, и придётся проверить все n элементов.`

**Average lookup в Hash Table:**  
`O(1). Мы считаем номер bucket по hash(key) % capacity и просматриваем только его. Пока load factor ограничен 0.75, в среднем в bucket лежит меньше одного элемента.`

**Worst-case lookup:**  
`O(n)`

**Почему worst-case может ухудшиться:**  
`если много ключей попадает в один bucket (плохая хеш-функция, неудачное распределение), цепочка вырождается в длинный список. Поиск в ней идёт линейно, то есть превращается в тот же linear_search.`

**Начальная capacity:**  
`8`

**Итоговая capacity после загрузки датасета:**  
`2097152(2²¹)`

**Итоговый load factor:**  
`≈ 0.4768 (1 000 000 / 2 097 152)`

**Когда происходит resize:**  
`Когда добавление нового ключа привело бы к (size+1)/capacity > 0.75. Тогда capacity увеличивается вдвое. Обновление существующего ключа resize не вызывает.`

**Почему при resize нужен rehash:**  
`Номер bucket считается как hash(key) % capacity. Когда capacity меняется, остаток от деления меняется, и один и тот же ключ должен лежать в другом bucket. Если просто увеличить список buckets, старые ключи окажутся не там, где get их ищет, и «потеряются». Поэтому все пары нужно заново разложить.`

**Сколько заняло построение индекса:**  
`8289.458 ms`

**Сколько занял linear search:**  
`933.542 ms`

**Сколько занял hash lookup:**  
`0.307 ms`

**Какую цену мы заплатили за быстрый поиск:**  
`Память: индекс хранит n пар (O(n) дополнительной памяти) и пустые buckets, так что в Python это сотни мегабайт на миллион записей. Время: индекс нужно один раз построить (секунды), это окупается лишь при достаточном числе запросов. Сложность: индекс нужно поддерживать в актуальном состоянии при вставках и удалениях. Ограничение: он ускоряет только точный поиск по ключу, а поиск по диапазону или в отсортированном порядке не поддерживает.`


---
# 10. Что будет дальше

Сегодня проект умеет только одно:

```text
быстро найти запись по уникальному ключу
```

На следующих занятиях будут появляться другие типы задач и новые структуры

Мы будем **добавлять их только тогда, когда появится конкретная проблема, которую текущая архитектура решает плохо**


---
# Дополнительно, если все основное готово

Добавьте метод:

```python
stats()
```

который возвращает или печатает:

```text
size
capacity
load_factor
количество пустых buckets
размер самого длинного bucket
```

После этого посмотрите статистику для индекса вашего датасета и коротко ответьте:

> Насколько равномерно элементы распределились по buckets?


ПРОДОЛЖЕНИЕ. SegmentTree

Порядок: created_at (оценки по времени)
Значение: rating
UPDATE i x - пользователь изменил оценку i-й по времени записи на x
QUERY L R - сумма оценок за период (делим на число записей, получаем среднюю)

In [ ]:
class SegmentTree:
    def __init__(self, a):
        self.n = len(a)
        self.a = list(a)
        self.tree = [0] * (4 * self.n)
        if self.n > 0:
            self._build(1, 0, self.n - 1)

    def _build(self, v, tl, tr):
        if tl == tr:
            self.tree[v] = self.a[tl]
            return
        tm = (tl + tr) // 2
        self._build(2 * v, tl, tm)
        self._build(2 * v + 1, tm + 1, tr)
        self.tree[v] = self.tree[2 * v] + self.tree[2 * v + 1]

    def query(self, l, r):
        if self.n == 0:
            return 0
        if not (0 <= l <= r < self.n):
            raise IndexError(f"Некорректный диапазон [{l}, {r}] для длины {self.n}")
        return self._query(1, 0, self.n - 1, l, r)

    def _query(self, v, tl, tr, l, r):
        if r < tl or tr < l:
            return 0
        if l <= tl and tr <= r:
            return self.tree[v]
        tm = (tl + tr) // 2
        return (self._query(2 * v, tl, tm, l, r)
                + self._query(2 * v + 1, tm + 1, tr, l, r))

    def update(self, pos, value):
        if not (0 <= pos < self.n):
            raise IndexError(f"Позиция {pos} вне массива длины {self.n}")
        self.a[pos] = value
        self._update(1, 0, self.n - 1, pos, value)

    def _update(self, v, tl, tr, pos, value):
        if tl == tr:
            self.tree[v] = value
            return
        tm = (tl + tr) // 2
        if pos <= tm:
            self._update(2 * v, tl, tm, pos, value)
        else:
            self._update(2 * v + 1, tm + 1, tr, pos, value)
        self.tree[v] = self.tree[2 * v] + self.tree[2 * v + 1]

In [ ]:
class SegmentTreeIndex:
    def __init__(self, df, order_col="created_at", value_col="rating"):
        self.df = df.sort_values(order_col).reset_index(drop=True)
        self.times = self.df[order_col].values
        values = self.df[value_col].tolist()

        start = perf_counter()
        self.st = SegmentTree(values)
        self.build_ms = (perf_counter() - start) * 1000

    def update(self, i, x):
        self.st.update(i, x)

    def range_sum(self, l, r):
        return self.st.query(l, r)

    def positions_between(self, start_date, end_date):
        l = np.searchsorted(self.times, np.datetime64(start_date), side="left")
        end = np.datetime64(end_date) + np.timedelta64(1, "D")
        r = np.searchsorted(self.times, end, side="left") - 1
        return int(l), int(r)

    def avg_between(self, start_date, end_date):
        l, r = self.positions_between(start_date, end_date)
        if l > r:
            return None
        return self.st.query(l, r) / (r - l + 1)

seg_index = SegmentTreeIndex(df)
print(f"Построено за {seg_index.build_ms:.0f} ms, элементов: {seg_index.st.n}")

Построено за 2287 ms, элементов: 1000000


In [ ]:
import random
random.seed(42)

N_OPS = 1000
n = seg_index.st.n

ops = []
for k in range(N_OPS):
    if k % 2 == 0:
        ops.append(("update", random.randrange(n), random.choice([0.5, 1.0, 5.0])))
    else:
        l = random.randrange(n)
        r = random.randrange(l, n)
        ops.append(("query", l, r))

n_updates = sum(1 for op in ops if op[0] == "update")
n_queries = N_OPS - n_updates


naive_values = seg_index.df["rating"].tolist()
start = perf_counter()
for op, x, y in ops:
    if op == "update":
        naive_values[x] = y
    else:
        sum(naive_values[x:y + 1])
naive_ms = (perf_counter() - start) * 1000

start = perf_counter()
for op, x, y in ops:
    if op == "update":
        seg_index.update(x, y)
    else:
        seg_index.range_sum(x, y)
seg_ms = (perf_counter() - start) * 1000

print(f"update: {n_updates}, query: {n_queries}")
print(f"Построение Segment Tree: {seg_index.build_ms:.1f} ms")
print(f"Наивно:       {naive_ms:.1f} ms")
print(f"Segment Tree: {seg_ms:.1f} ms")
print(f"Ускорение: {naive_ms / seg_ms:.1f}x")

update: 500, query: 500
Построение Segment Tree: 2287.0 ms
Наивно:       2715.5 ms
Segment Tree: 19.6 ms
Ускорение: 138.5x
